# Análisis estadístico definitivo — VGG-16 y TrashNet

Este cuaderno analiza el experimento principal PNG con 2527 imágenes, la estabilidad del subconjunto balanceado de 822 imágenes, la comparación controlada JPG–PNG y la asociación exploratoria entre energía de gradiente periférica y cambio representacional en Block5.

Las familias de corrección múltiple se definen explícitamente. El subconjunto 822 no se trata como muestra independiente del conjunto de 2527 porque está contenido en él.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
from pathlib import Path
import itertools, json, warnings
import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon, kruskal, mannwhitneyu, spearmanr, rankdata
import matplotlib.pyplot as plt

BASE_DIR = Path('/content/gdrive/MyDrive/2026-0')
FULL_CSV = BASE_DIR / 'Resultados_VGG16_TrashNet_completo_v2_resize224' / 'embeddings_metricas_VGG16_TrashNet_completo_v2.csv'
FORMAT_CSV = BASE_DIR / 'Resultados_comparacion_controlada_822_JPG_PNG' / 'comparacion_controlada_822_JPG_PNG.csv'
AUDIT_CSV = BASE_DIR / 'Resultados_comparacion_controlada_822_JPG_PNG' / 'auditoria_originales_822.csv'
ORIGINAL_ROOT = BASE_DIR / 'DatasetOriginalCompleto'
OUTPUT_ROOT = BASE_DIR / 'Resultados_estadisticos_definitivos_VGG16'
FIGURE_ROOT = OUTPUT_ROOT / 'figuras'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

for path in [FULL_CSV, FORMAT_CSV, AUDIT_CSV, ORIGINAL_ROOT]:
    if not path.exists(): raise FileNotFoundError(f'No existe: {path}')

LEVEL_ORDER = ['low', 'medium', 'high']
BLOCK_ORDER = ['Block1', 'Block2', 'Block3', 'Block4', 'Block5']
CLASS_ORDER = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
METRICS = ['cosine_similarity', 'relative_euclidean_distance']
ALPHA = 0.05
full = pd.read_csv(FULL_CSV)
fmt = pd.read_csv(FORMAT_CSV)
audit = pd.read_csv(AUDIT_CSV)
print('Principal PNG:', full.shape)
print('Comparación JPG–PNG:', fmt.shape)
print('Auditoría originales:', audit.shape)

## 1. Funciones estadísticas y auditoría

In [ ]:
def holm_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p)
    adjusted = np.empty(len(p), dtype=float)
    running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (len(p) - rank) * p[idx])
        adjusted[idx] = min(running, 1.0)
    return adjusted

def paired_rank_biserial(differences):
    d = np.asarray(differences, dtype=float)
    d = d[d != 0]
    if len(d) == 0: return 0.0
    ranks = rankdata(np.abs(d), method='average')
    positive = ranks[d > 0].sum(); negative = ranks[d < 0].sum()
    return float((positive - negative) / (positive + negative))

def safe_wilcoxon(a, b):
    a = np.asarray(a, float); b = np.asarray(b, float)
    if np.allclose(a, b): return 0.0, 1.0
    result = wilcoxon(a, b, zero_method='wilcox', alternative='two-sided', method='auto')
    return float(result.statistic), float(result.pvalue)

main_key = ['class', 'image_id', 'transformation', 'level', 'block']
if len(full) != 151620 or full.duplicated(main_key).any(): raise ValueError('El CSV principal no supera conteo/unicidad.')
if len(fmt) != 49320 or fmt.duplicated(main_key).any(): raise ValueError('El CSV JPG–PNG no supera conteo/unicidad.')
if full.isna().any().any() or fmt.isna().any().any(): raise ValueError('Existen valores ausentes.')
if len(audit) != 822 or not audit['pixel_equal'].all(): raise ValueError('La identidad de las 822 imágenes no está confirmada.')
numeric_full = full.select_dtypes(include=np.number)
numeric_fmt = fmt.select_dtypes(include=np.number)
if not np.isfinite(numeric_full.to_numpy()).all() or not np.isfinite(numeric_fmt.to_numpy()).all(): raise ValueError('Existen valores no finitos.')
print('AUDITORÍA APROBADA')

## 2. Descriptivos micro, macro y estabilidad 822–2527

In [ ]:
group_cols = ['transformation', 'level', 'value', 'block']
micro = full.groupby(group_cols, observed=True).agg(
    n=('image_id','size'), cosine_mean=('cosine_similarity','mean'), cosine_sd=('cosine_similarity','std'),
    cosine_median=('cosine_similarity','median'), relative_mean=('relative_euclidean_distance','mean'),
    relative_sd=('relative_euclidean_distance','std'), relative_median=('relative_euclidean_distance','median')
).reset_index()
class_means = full.groupby(group_cols + ['class'], observed=True)[METRICS].mean().reset_index()
macro = class_means.groupby(group_cols, observed=True).agg(
    classes=('class','nunique'), cosine_macro_mean=('cosine_similarity','mean'),
    cosine_macro_sd_between_classes=('cosine_similarity','std'),
    relative_macro_mean=('relative_euclidean_distance','mean'),
    relative_macro_sd_between_classes=('relative_euclidean_distance','std')
).reset_index()
subset = full[full['in_balanced_822']].copy()
subset_summary = subset.groupby(group_cols, observed=True)[METRICS].mean().reset_index().rename(columns={
    'cosine_similarity':'cosine_822_mean', 'relative_euclidean_distance':'relative_822_mean'})
stability = micro.merge(subset_summary, on=group_cols, validate='one_to_one')
stability['delta_cosine_822_minus_full'] = stability['cosine_822_mean'] - stability['cosine_mean']
stability['delta_relative_822_minus_full'] = stability['relative_822_mean'] - stability['relative_mean']
stability.to_csv(OUTPUT_ROOT / '01_estabilidad_822_vs_2527.csv', index=False)
micro.to_csv(OUTPUT_ROOT / '02_descriptivos_micro_2527.csv', index=False)
macro.to_csv(OUTPUT_ROOT / '03_descriptivos_macro_clases.csv', index=False)
class_means.to_csv(OUTPUT_ROOT / '04_medias_por_clase.csv', index=False)
stability_report = []
for full_col, sub_col, label in [('cosine_mean','cosine_822_mean','cosine'),('relative_mean','relative_822_mean','relative')]:
    rho, p = spearmanr(stability[full_col], stability[sub_col])
    stability_report.append({'metric':label, 'spearman_rho_across_60_cells':rho, 'p_value':p,
        'mean_absolute_difference':np.mean(np.abs(stability[sub_col]-stability[full_col])),
        'maximum_absolute_difference':np.max(np.abs(stability[sub_col]-stability[full_col]))})
pd.DataFrame(stability_report).to_csv(OUTPUT_ROOT / '05_resumen_estabilidad_822_vs_2527.csv', index=False)
display(micro.head())

## 3. Intensidades

Familia post hoc: las tres comparaciones de niveles dentro de una transformación × bloque × métrica. Los 20 omnibus de cada métrica también reciben Holm.

In [ ]:
intensity_omnibus = []; intensity_posthoc = []
for metric in METRICS:
    for (transformation, block), g in full.groupby(['transformation','block'], observed=True):
        wide = g.pivot(index=['class','image_id'], columns='level', values=metric).dropna()
        wide = wide[LEVEL_ORDER]
        chi2, p = friedmanchisquare(*(wide[level] for level in LEVEL_ORDER))
        n, k = len(wide), len(LEVEL_ORDER); kendall_w = chi2 / (n * (k - 1))
        intensity_omnibus.append({'metric':metric,'transformation':transformation,'block':block,'n':n,'k':k,
            'friedman_chi2':chi2,'df':k-1,'p_value':p,'kendall_w':kendall_w})
        family=[]
        for a,b in itertools.combinations(LEVEL_ORDER,2):
            stat,pair_p=safe_wilcoxon(wide[a],wide[b]); diff=wide[b]-wide[a]
            family.append({'metric':metric,'transformation':transformation,'block':block,'level_a':a,'level_b':b,
                'n':n,'wilcoxon_w':stat,'p_value':pair_p,'median_a':wide[a].median(),'median_b':wide[b].median(),
                'median_difference_b_minus_a':diff.median(),'paired_rank_biserial':paired_rank_biserial(diff)})
        adjusted=holm_adjust([x['p_value'] for x in family])
        for row,adj in zip(family,adjusted): row['p_holm_within_family']=adj
        intensity_posthoc.extend(family)
intensity_omnibus=pd.DataFrame(intensity_omnibus)
for metric, idx in intensity_omnibus.groupby('metric').groups.items():
    intensity_omnibus.loc[idx,'p_holm_across_omnibus'] = holm_adjust(intensity_omnibus.loc[idx,'p_value'])
intensity_omnibus['significant_holm'] = intensity_omnibus['p_holm_across_omnibus'] < ALPHA
intensity_omnibus.to_csv(OUTPUT_ROOT / '06_friedman_intensidades_kendall_w.csv', index=False)
pd.DataFrame(intensity_posthoc).to_csv(OUTPUT_ROOT / '07_wilcoxon_intensidades_holm.csv', index=False)
display(intensity_omnibus.head())

## 4. Bloques

Familia post hoc: las diez comparaciones de bloques dentro de una transformación × nivel × métrica. Los 12 omnibus de cada métrica reciben Holm.

In [ ]:
block_omnibus=[]; block_posthoc=[]
for metric in METRICS:
    for (transformation, level),g in full.groupby(['transformation','level'], observed=True):
        wide=g.pivot(index=['class','image_id'],columns='block',values=metric).dropna()[BLOCK_ORDER]
        chi2,p=friedmanchisquare(*(wide[b] for b in BLOCK_ORDER)); n,k=len(wide),len(BLOCK_ORDER)
        block_omnibus.append({'metric':metric,'transformation':transformation,'level':level,'n':n,'k':k,
            'friedman_chi2':chi2,'df':k-1,'p_value':p,'kendall_w':chi2/(n*(k-1))})
        family=[]
        for a,b in itertools.combinations(BLOCK_ORDER,2):
            stat,pair_p=safe_wilcoxon(wide[a],wide[b]); diff=wide[b]-wide[a]
            family.append({'metric':metric,'transformation':transformation,'level':level,'block_a':a,'block_b':b,
                'n':n,'wilcoxon_w':stat,'p_value':pair_p,'median_a':wide[a].median(),'median_b':wide[b].median(),
                'median_difference_b_minus_a':diff.median(),'paired_rank_biserial':paired_rank_biserial(diff)})
        adjusted=holm_adjust([x['p_value'] for x in family])
        for row,adj in zip(family,adjusted): row['p_holm_within_family']=adj
        block_posthoc.extend(family)
block_omnibus=pd.DataFrame(block_omnibus)
for metric,idx in block_omnibus.groupby('metric').groups.items():
    block_omnibus.loc[idx,'p_holm_across_omnibus']=holm_adjust(block_omnibus.loc[idx,'p_value'])
block_omnibus['significant_holm']=block_omnibus['p_holm_across_omnibus']<ALPHA
block_omnibus.to_csv(OUTPUT_ROOT / '08_friedman_bloques_kendall_w.csv',index=False)
pd.DataFrame(block_posthoc).to_csv(OUTPUT_ROOT / '09_wilcoxon_bloques_holm.csv',index=False)
display(block_omnibus.head())

## 5. Diferencias entre clases

Familia post hoc: las quince comparaciones entre clases dentro de una transformación × nivel × bloque × métrica. Los 60 omnibus de cada métrica reciben Holm.

In [ ]:
class_omnibus=[]; class_posthoc=[]
for metric in METRICS:
    for (transformation,level,block),g in full.groupby(['transformation','level','block'],observed=True):
        arrays=[g.loc[g['class']==c,metric].to_numpy() for c in CLASS_ORDER]
        h,p=kruskal(*arrays); n=len(g); k=len(CLASS_ORDER); epsilon_sq=max(0.0,(h-k+1)/(n-k))
        class_omnibus.append({'metric':metric,'transformation':transformation,'level':level,'block':block,
            'n':n,'k':k,'kruskal_h':h,'df':k-1,'p_value':p,'epsilon_squared':epsilon_sq})
        family=[]
        for a,b in itertools.combinations(CLASS_ORDER,2):
            x=g.loc[g['class']==a,metric].to_numpy(); y=g.loc[g['class']==b,metric].to_numpy()
            result=mannwhitneyu(x,y,alternative='two-sided',method='auto')
            rbc=2*result.statistic/(len(x)*len(y))-1
            family.append({'metric':metric,'transformation':transformation,'level':level,'block':block,
                'class_a':a,'class_b':b,'n_a':len(x),'n_b':len(y),'mann_whitney_u':result.statistic,
                'p_value':result.pvalue,'median_a':np.median(x),'median_b':np.median(y),'rank_biserial_a_vs_b':rbc})
        adjusted=holm_adjust([x['p_value'] for x in family])
        for row,adj in zip(family,adjusted): row['p_holm_within_family']=adj
        class_posthoc.extend(family)
class_omnibus=pd.DataFrame(class_omnibus)
for metric,idx in class_omnibus.groupby('metric').groups.items():
    class_omnibus.loc[idx,'p_holm_across_omnibus']=holm_adjust(class_omnibus.loc[idx,'p_value'])
class_omnibus['significant_holm']=class_omnibus['p_holm_across_omnibus']<ALPHA
class_omnibus.to_csv(OUTPUT_ROOT / '10_kruskal_wallis_clases.csv',index=False)
pd.DataFrame(class_posthoc).to_csv(OUTPUT_ROOT / '11_mann_whitney_clases_holm.csv',index=False)
display(class_omnibus.head())

## 6. Comparación secundaria JPG–PNG

Familia Holm: las 60 combinaciones transformación × nivel × bloque dentro de cada métrica. Se reportan magnitud, dirección y similitud directa JPG–PNG.

In [ ]:
format_rows=[]
format_specs=[
    ('cosine','cosine_jpg','cosine_png','delta_cosine_png_minus_jpg'),
    ('relative_euclidean','relative_euclidean_jpg','relative_euclidean_png','delta_relative_png_minus_jpg'),
    ('euclidean','euclidean_jpg','euclidean_png','delta_euclidean_png_minus_jpg')]
for metric,jpg_col,png_col,delta_col in format_specs:
    for (transformation,level,block),g in fmt.groupby(['transformation','level','block'],observed=True):
        stat,p=safe_wilcoxon(g[jpg_col],g[png_col]); diff=g[png_col]-g[jpg_col]
        format_rows.append({'metric':metric,'transformation':transformation,'level':level,'block':block,'n':len(g),
            'jpg_mean':g[jpg_col].mean(),'png_mean':g[png_col].mean(),'mean_difference_png_minus_jpg':diff.mean(),
            'median_difference_png_minus_jpg':diff.median(),'mean_absolute_difference':np.mean(np.abs(diff)),
            'wilcoxon_w':stat,'p_value':p,'paired_rank_biserial':paired_rank_biserial(diff),
            'direct_jpg_png_cosine_mean':g['cosine_jpg_vs_png'].mean(),
            'direct_jpg_png_cosine_median':g['cosine_jpg_vs_png'].median(),
            'direct_jpg_png_relative_mean':g['relative_euclidean_jpg_vs_png'].mean()})
format_results=pd.DataFrame(format_rows)
for metric,idx in format_results.groupby('metric').groups.items():
    format_results.loc[idx,'p_holm_across_60_cells']=holm_adjust(format_results.loc[idx,'p_value'])
format_results['significant_holm']=format_results['p_holm_across_60_cells']<ALPHA
format_results.to_csv(OUTPUT_ROOT / '12_comparacion_pareada_JPG_PNG.csv',index=False)
display(format_results.head())

## 7. Gradiente periférico y zoom

La energía periférica se define como la proporción de magnitud Sobel situada fuera del recorte central correspondiente a cada zoom. Se relaciona mediante Spearman con distancia relativa y 1−coseno en Block5. El resultado es asociativo, no causal.

In [ ]:
from PIL import Image
from scipy.ndimage import sobel
from tqdm.auto import tqdm

PERIPHERAL_CSV=OUTPUT_ROOT/'13_energia_gradiente_periferica.csv'
ZOOM_FACTORS={'low':1.10,'medium':1.25,'high':1.50}
if PERIPHERAL_CSV.exists():
    peripheral=pd.read_csv(PERIPHERAL_CSV)
else:
    items=full[['class','image_id','original_filename']].drop_duplicates().sort_values(['class','image_id'])
    rows=[]
    for class_name,image_id,original_filename in tqdm(items.itertuples(index=False,name=None),total=len(items),desc='Gradiente periférico'):
        path=ORIGINAL_ROOT/class_name/original_filename
        with Image.open(path) as im: gray=np.asarray(im.convert('L'),dtype=np.float32)/255.0
        gx=sobel(gray,axis=1,mode='reflect'); gy=sobel(gray,axis=0,mode='reflect'); magnitude=np.hypot(gx,gy)
        total_energy=float(magnitude.sum())+1e-12; h,w=gray.shape
        for level,factor in ZOOM_FACTORS.items():
            crop_w=max(1,int(w/factor)); crop_h=max(1,int(h/factor))
            left=(w-crop_w)//2; top=(h-crop_h)//2
            mask=np.ones((h,w),dtype=bool); mask[top:top+crop_h,left:left+crop_w]=False
            rows.append({'class':class_name,'image_id':image_id,'original_filename':original_filename,
                'level':level,'zoom_factor':factor,'removed_area_fraction':mask.mean(),
                'peripheral_gradient_energy_ratio':float(magnitude[mask].sum()/total_energy)})
    peripheral=pd.DataFrame(rows); peripheral.to_csv(PERIPHERAL_CSV,index=False)
zoom=full[(full['transformation']=='zoom')&(full['block']=='Block5')].copy()
zoom['one_minus_cosine']=1-zoom['cosine_similarity']
zoom=zoom.merge(peripheral,on=['class','image_id','original_filename','level'],validate='one_to_one')
zoom.to_csv(OUTPUT_ROOT/'14_zoom_block5_con_gradiente.csv',index=False)
spearman_rows=[]
for outcome in ['relative_euclidean_distance','one_minus_cosine']:
    for level,g in zoom.groupby('level',observed=True):
        for stratum in ['overall']+CLASS_ORDER:
            x=g if stratum=='overall' else g[g['class']==stratum]
            rho,p=spearmanr(x['peripheral_gradient_energy_ratio'],x[outcome])
            spearman_rows.append({'outcome':outcome,'level':level,'stratum':stratum,'n':len(x),'spearman_rho':rho,'p_value':p})
spearman_results=pd.DataFrame(spearman_rows)
for outcome,idx in spearman_results.groupby('outcome').groups.items():
    spearman_results.loc[idx,'p_holm_across_21_tests']=holm_adjust(spearman_results.loc[idx,'p_value'])
spearman_results['significant_holm']=spearman_results['p_holm_across_21_tests']<ALPHA
spearman_results.to_csv(OUTPUT_ROOT/'15_spearman_gradiente_periferico_block5.csv',index=False)
display(spearman_results.head())

## 8. Figuras y resumen de auditoría

In [ ]:
colors={'low':'#4C78A8','medium':'#F58518','high':'#E45756'}
panel_labels = ['(a)', '(b)', '(c)', '(d)']
for metric,ylabel,filename in [('cosine_mean','Similitud coseno media','16_coseno_por_bloque.png'),('relative_mean','Distancia euclidiana relativa media','17_distancia_relativa_por_bloque.png')]:
    fig,axes=plt.subplots(2,2,figsize=(12,8),sharex=True); axes=axes.ravel()
    for panel_idx,(ax,(transformation,g)) in enumerate(zip(axes,micro.groupby('transformation',sort=True))):
        for level in LEVEL_ORDER:
            z=g[g['level']==level].set_index('block').reindex(BLOCK_ORDER)
            ax.plot(BLOCK_ORDER,z[metric],marker='o',label=level,color=colors[level])
        ax.set_title(f'{panel_labels[panel_idx]} {transformation.capitalize()}', fontweight='bold'); ax.set_ylabel(ylabel); ax.grid(alpha=.25)
    axes[0].legend(title='Intensidad'); fig.tight_layout(); fig.savefig(FIGURE_ROOT/filename,dpi=300,bbox_inches='tight'); plt.show()

format_order = pd.MultiIndex.from_product([['brightness','contrast','rotation','zoom'], LEVEL_ORDER], names=['transformation','level'])
pivot=format_results[format_results['metric']=='cosine'].pivot(index=['transformation','level'],columns='block',values='mean_difference_png_minus_jpg').reindex(index=format_order,columns=BLOCK_ORDER)
fig,ax=plt.subplots(figsize=(9,6)); im=ax.imshow(pivot.values,aspect='auto',cmap='coolwarm')
ax.set_xticks(range(len(BLOCK_ORDER)),BLOCK_ORDER); ax.set_yticks(range(len(pivot)),[f'{a} {b}' for a,b in pivot.index])
ax.set_title('Diferencia de similitud coseno: PNG − JPG'); fig.colorbar(im,ax=ax,label='Diferencia media')
fig.tight_layout(); fig.savefig(FIGURE_ROOT/'18_diferencia_coseno_PNG_menos_JPG.png',dpi=300,bbox_inches='tight'); plt.show()

audit_summary={
 'principal_rows':int(len(full)),'principal_unique_images':int(full[['class','image_id']].drop_duplicates().shape[0]),
 'balanced_rows':int(full['in_balanced_822'].sum()),'format_rows':int(len(fmt)),
 'originals_verified_identical':int(audit['pixel_equal'].sum()),
 'multiple_comparison_families':{
  'intensity_posthoc':'3 level pairs within transformation × block × metric',
  'block_posthoc':'10 block pairs within transformation × level × metric',
  'class_posthoc':'15 class pairs within transformation × level × block × metric',
  'format':'60 transformation × level × block cells within each metric',
  'peripheral_gradient':'21 level × stratum tests within each outcome'},
 'interpretive_limits':['822 is nested within 2527; no independent-samples test is used between them',
  'peripheral gradient analysis is associative, not causal','statistical significance must be interpreted with effect sizes']
}
with (OUTPUT_ROOT/'00_auditoria_y_decisiones_metodologicas.json').open('w',encoding='utf-8') as f: json.dump(audit_summary,f,indent=2,ensure_ascii=False)
print('ANÁLISIS FINALIZADO')
print('Resultados:',OUTPUT_ROOT)